In [ ]:
# 1. Load datasets from /data
df = pd.read_csv(CSV_PATH)
df["year"] = df["year"].astype(int)
df["area_restored"] = df["area_restored"].astype(int)
df["carbon"] = df["carbon"].astype(int)

# Restoration type by region (aligned with dashboard logic)
def restoration_type_for_region(region: str) -> str:
    mapping = {
        "Pará": "Plantio de nativas",
        "Acre": "Regeneração natural",
        "Rondônia": "Sistema agroflorestal",
        "Amazonas": "Regeneração natural",
    }
    return mapping.get(region, "Não informado")

df["restoration_type"] = df["region"].map(restoration_type_for_region)
df

In [ ]:
# 2. Advanced scientific metrics

# Carbon sequestration efficiency per hectare (tCO2e/ha)
df["carbon_per_hectare"] = (df["carbon"] / df["area_restored"]).round(2)

# Year-over-year area change for growth rate (per region)
df_sorted = df.sort_values(["region", "year"])
df_sorted["area_prev"] = df_sorted.groupby("region")["area_restored"].shift(1)
df_sorted["restoration_growth_rate"] = (
    (df_sorted["area_restored"] - df_sorted["area_prev"]) / df_sorted["area_prev"].replace(0, 1)
).round(4)
df_sorted["restoration_growth_rate_pct"] = (df_sorted["restoration_growth_rate"] * 100).round(1)
df = df_sorted.drop(columns=["area_prev"])
df

In [ ]:
# Restoration trends (by year, region; growth rate)
restoration_trends = (
    df.groupby(["year", "region"])
    .agg(
        area_restored=("area_restored", "sum"),
        carbon=("carbon", "sum"),
        carbon_per_hectare=("carbon_per_hectare", "mean"),
        restoration_growth_rate_pct=("restoration_growth_rate_pct", "last"),
    )
    .reset_index()
)
restoration_trends = restoration_trends.where(pd.notna(restoration_trends), None)
restoration_trends_list = restoration_trends.to_dict(orient="records")

# Carbon efficiency (region, year, carbon_per_hectare)
carbon_efficiency = (
    df.groupby(["region", "year"])
    .agg(carbon_per_hectare=("carbon_per_hectare", "mean"), area_restored=("area_restored", "sum"), carbon=("carbon", "sum"))
    .reset_index()
)
carbon_efficiency_list = carbon_efficiency.to_dict(orient="records")

# Region performance (total area, carbon, mean carbon/ha, growth)
region_totals = df.groupby("region").agg(
    total_area_restored=("area_restored", "sum"),
    total_carbon=("carbon", "sum"),
    mean_carbon_per_hectare=("carbon_per_hectare", "mean"),
).round(2)
region_totals = region_totals.reset_index()
region_performance_list = region_totals.to_dict(orient="records")

# Method contribution (percentage of area by restoration type)
total_area = df["area_restored"].sum()
method_area = df.groupby("restoration_type")["area_restored"].sum()
method_contribution = (method_area / total_area * 100).round(1).reset_index()
method_contribution.columns = ["restoration_type", "percentage"]
method_contribution_list = method_contribution.to_dict(orient="records")

# Biodiversity index approximation (0-1 scale from carbon/ha normalized)
max_cph = df["carbon_per_hectare"].max()
df["biodiversity_index"] = (df["carbon_per_hectare"] / max_cph if max_cph else 0).round(3)
biodiversity_by_region_year = df[["region", "year", "biodiversity_index", "carbon_per_hectare"]].to_dict(orient="records")

In [ ]:
# 3. Detect trends and build scientific insights
insights = []

# Highest carbon density (per hectare)
if not carbon_efficiency.empty:
    best = carbon_efficiency.loc[carbon_efficiency["carbon_per_hectare"].idxmax()]
    insights.append({
        "id": "carbon_density",
        "text_pt": f"{best['region']} apresenta o maior sequestro de carbono por hectare em {int(best['year'])} ({best['carbon_per_hectare']:.1f} tCO₂e/ha).",
        "text_en": f"{best['region']} shows the highest carbon sequestration per hectare in {int(best['year'])} ({best['carbon_per_hectare']:.1f} tCO₂e/ha).",
    })

# Restoration area trend
years = sorted(df["year"].unique())
if len(years) >= 2:
    insights.append({
        "id": "area_trend",
        "text_pt": f"A área em restauração aumentou de forma contínua entre {years[0]} e {years[-1]}.",
        "text_en": f"Restoration area increased steadily between {years[0]} and {years[-1]}.",
    })

# Method with highest contribution
if not method_contribution.empty:
    top_method = method_contribution.loc[method_contribution["percentage"].idxmax()]
    insights.append({
        "id": "method_contribution",
        "text_pt": f"{top_method['restoration_type']} representa a maior contribuição à área restaurada ({top_method['percentage']:.0f}%).",
        "text_en": f"{top_method['restoration_type']} represents the largest contribution to restored area ({top_method['percentage']:.0f}%).",
    })

# Region with highest total restoration in latest year
latest_year = df["year"].max()
latest = df[df["year"] == latest_year]
if not latest.empty:
    top_region = latest.loc[latest["area_restored"].idxmax()]
    insights.append({
        "id": "leading_region",
        "text_pt": f"Em {int(latest_year)}, {top_region['region']} registrou a maior área restaurada em um único ano ({int(top_region['area_restored'])} ha).",
        "text_en": f"In {int(latest_year)}, {top_region['region']} recorded the largest restored area in a single year ({int(top_region['area_restored'])} ha).",
    })

In [ ]:
# 4. Export processed datasets for FastAPI
def to_json_safe(obj):
    if hasattr(obj, "item"):  # numpy scalar
        return float(obj) if isinstance(obj, (float,)) else int(obj)
    if pd.isna(obj):
        return None
    return obj

def clean_records(records):
    return [{k: to_json_safe(v) for k, v in r.items()} for r in records]

(OUTPUT_DIR / "restoration_trends.json").write_text(
    json.dumps(clean_records(restoration_trends_list), indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "carbon_efficiency.json").write_text(
    json.dumps(clean_records(carbon_efficiency_list), indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "region_performance.json").write_text(
    json.dumps(clean_records(region_performance_list), indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "method_contribution.json").write_text(
    json.dumps(clean_records(method_contribution_list), indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "scientific_insights.json").write_text(
    json.dumps(insights, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Exported:", list(OUTPUT_DIR.glob("*.json")))